In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# CNN + DEVIGN : FINAL ONE-CELL KAGGLE CODE
# Dataset: devignx-cnn
# ============================================================

# --------------------
# Imports
# --------------------
import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import platform
import psutil
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# HARDWARE INFO
# ============================================================

hardware_info = {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "None",
    "torch_version": torch.__version__,
    "ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
    "os": platform.system()
}

print("\nHardware Info:", hardware_info)

# ============================================================
# DATASET PATH (✅ FIXED)
# ============================================================

DATASET_PATH = "/kaggle/input/devignx-cnn"

print("\nFiles in dataset:")
for f in os.listdir(DATASET_PATH):
    print(" -", f)

# ============================================================
# LOAD DEVIGN CSV
# ============================================================

def load_devign_csv(path):
    df = pd.read_csv(path)

    if "func" in df.columns:
        df["code"] = df["func"]
    elif "code" not in df.columns:
        raise ValueError("No code column found")

    if "target" in df.columns:
        df["label"] = df["target"]
    elif "label" not in df.columns:
        raise ValueError("No label column found")

    df = df[["code", "label"]]
    df["label"] = df["label"].astype(int)
    return df

print("\nLoading Devign dataset...")

train_df = load_devign_csv(f"{DATASET_PATH}/devignx_train.csv")
val_df   = load_devign_csv(f"{DATASET_PATH}/Devignx_validation.csv")
test_df  = load_devign_csv(f"{DATASET_PATH}/devignx_test.csv")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ============================================================
# TOKENIZER (FOR CNN INPUT)
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
MAX_LEN = 256

class DevignDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.codes[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = DevignDataset(train_df)
val_ds   = DevignDataset(val_df)
test_ds  = DevignDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

# ============================================================
# CNN MODEL
# ============================================================

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embed_dim, 128, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(embed_dim, 128, kernel_size=7, padding=3)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(128 * 3, num_classes)

    def forward(self, x):
        x = self.embedding(x)          # (B, L, E)
        x = x.permute(0, 2, 1)         # (B, E, L)

        x1 = torch.relu(self.conv1(x))
        x2 = torch.relu(self.conv2(x))
        x3 = torch.relu(self.conv3(x))

        x1 = torch.max(x1, dim=2)[0]
        x2 = torch.max(x2, dim=2)[0]
        x3 = torch.max(x3, dim=2)[0]

        x = torch.cat([x1, x2, x3], dim=1)
        x = self.dropout(x)
        return self.fc(x)

model = CNNClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128
).to(device)

# ============================================================
# TRAINING SETUP
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 6

# ============================================================
# TRAIN LOOP
# ============================================================

print("\nStarting CNN training...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f}")

# ============================================================
# EVALUATION
# ============================================================

print("\nEvaluating on test set...")

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        preds = torch.argmax(model(input_ids), dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== CNN DEVIGN RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)

# ============================================================
# SAVE RESULTS
# ============================================================

results = {
    "dataset": "Devign",
    "model": "CNN",
    "epochs": EPOCHS,
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr)
    },
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "hardware": hardware_info,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/CNN_Devign_results.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


Device: cuda
GPU: Tesla T4

Hardware Info: {'gpu': 'Tesla T4', 'cuda_version': '12.6', 'torch_version': '2.8.0+cu126', 'ram_gb': 31.35, 'os': 'Linux'}

Files in dataset:
 - Devignx_validation.csv
 - devignx_test.csv
 - devignx_train.csv

Loading Devign dataset...
Train: (19122, 2)
Val  : (2732, 2)
Test : (2732, 2)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


Starting CNN training...
Epoch 1/6 | Train Loss: 0.7560
Epoch 2/6 | Train Loss: 0.6995
Epoch 3/6 | Train Loss: 0.6943
Epoch 4/6 | Train Loss: 0.6818
Epoch 5/6 | Train Loss: 0.6711
Epoch 6/6 | Train Loss: 0.6614

Evaluating on test set...

===== CNN DEVIGN RESULTS =====
Accuracy : 0.5629575402635432
Precision: 0.5406162464985994
Recall   : 0.3083067092651757
F1 Score : 0.392675483214649
FPR      : 0.22162162162162163
Confusion Matrix: 1152 328 866 386

Results saved to: /kaggle/working/CNN_Devign_results.json
